In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("msambare/fer2013")

print("Path to dataset files:", path)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 60.3M/60.3M [00:34<00:00, 1.85MB/s]

Extracting files...


Path to dataset files: C:\Users\donel\.cache\kagglehub\datasets\msambare\fer2013\versions\1


In [3]:
import os
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision.models import vit_b_16
from tqdm import tqdm

# Configuration
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 1e-4
NUM_CLASSES = 7  # All expressions in FER2013
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Fixed paths (use raw strings or escape slashes)
TRAIN_DIR = r"C:\Users\donel\.cache\kagglehub\datasets\msambare\fer2013\versions\1\train"
TEST_DIR = r"C:\Users\donel\.cache\kagglehub\datasets\msambare\fer2013\versions\1\test"

# Transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

# Datasets and Loaders
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# Vision Transformer Model
class ViTEmotionClassifier(nn.Module):
    def __init__(self, num_classes):
        super(ViTEmotionClassifier, self).__init__()
        self.model = vit_b_16(pretrained=True)
        self.model.heads = nn.Linear(self.model.heads.in_features, num_classes)

    def forward(self, x):
        return self.model(x)

model = ViTEmotionClassifier(num_classes=NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Training
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"[Epoch {epoch+1}] Training Loss: {avg_loss:.4f}")

# Save model
torch.save(model.state_dict(), "vit_emotion_model_all.pth")
print("[✅] Model saved as vit_emotion_model_all.pth")


C:\Users\donel\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\donel\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ViT_B_16_Weights.IMAGENET1K_V1`. You can also use `weights=ViT_B_16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to C:\Users\donel/.cache\torch\hub\checkpoints\vit_b_16-c867db91.pth
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 330M/330M [03:54<00:00, 1.48MB/s]


AttributeError: 'Sequential' object has no attribute 'in_features'

In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision.models import vit_b_16
from tqdm import tqdm

# Configuration
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 1e-4
NUM_CLASSES = 7  # All expressions in FER2013
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Paths (FIXED with raw strings for Windows)
TRAIN_DIR = r"C:\Users\donel\.cache\kagglehub\datasets\msambare\fer2013\versions\1\train"
TEST_DIR = r"C:\Users\donel\.cache\kagglehub\datasets\msambare\fer2013\versions\1\test"

# Transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

# Datasets and Loaders
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# Vision Transformer Model
class ViTEmotionClassifier(nn.Module):
    def __init__(self, num_classes):
        super(ViTEmotionClassifier, self).__init__()
        self.model = vit_b_16(pretrained=True)
        
        # Access in_features safely from the first layer of the classifier
        in_features = self.model.heads[0].in_features
        self.model.heads = nn.Sequential(
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.model(x)

# Initialize model
model = ViTEmotionClassifier(num_classes=NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Training loop
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"[Epoch {epoch+1}] Training Loss: {avg_loss:.4f}")

# Save model
torch.save(model.state_dict(), "vit_emotion_model_all.pth")
print("[✅] Model saved as vit_emotion_model_all.pth")


Epoch 1/10:  11%|███████████▏                                                                                       | 101/898 [6:13:08<20:03:38, 90.61s/it]